# CM_TM_06 - Match Involvement of Goal Scorers

## Libraries

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go


## Import Files

In [5]:
# 2. Import File
path_to_file = "https://raw.githubusercontent.com/cesarlarasantana/VDS_2526_G04_Football/refs/heads/main/Datasets"

df_country = pd.read_csv(path_to_file+"/Country.csv")
df_match_goals = pd.read_csv(path_to_file+"/Match_Goals.csv")
df_match_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")
df_match_cards = pd.read_csv(path_to_file+"/Match_Cards.csv")
df_team = pd.read_csv(path_to_file+"/Team.csv")
df_match = pd.read_csv(path_to_file+"/Match.csv")

df_player = pd.read_csv(path_to_file+"/Player.csv")
df_player_att = pd.read_csv(path_to_file+"/Player_Attributes.csv")
df_cross = pd.read_csv(path_to_file+"/Match_Cross.csv")
df_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_shots_off = pd.read_csv(path_to_file+"/Match_Shots_Off.csv")

C:\Users\pci\AppData\Local\Temp\ipykernel_9244\107979182.py:7: DtypeWarning: Columns (0: player1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")


## Data Preparation

In [6]:
''' Data Preparation '''

## Create dataframe with information on players ##
info_on_player_list = []
# Determening the players position with y coordinates 
for _, row in df_match.iterrows():
    season = row["season"]
    # Home players
    for i in range(1, 12):
        player_col = f"home_player_{i}"
        y_col = f"home_player_Y{i}"
        player_id = row[player_col]
        if pd.notna(player_id):
            y_value = row[y_col]
            if pd.isna(y_value):
                position = None
            elif y_value <= 1:
                position = "Goalkeeper"
            elif y_value <= 3:
                position = "Defender"
            elif y_value <= 8:
                position = "Midfielder"
            else:
                position = "Forwarder"
            info_on_player_list.append({"season": season,
                                        "team_api_id": row["home_team_api_id"],
                                        "player_api_id": player_id,
                                        "position": position})
    # Away players
    for i in range(1, 12):
        player_col = f"away_player_{i}"
        y_col = f"away_player_Y{i}"
        player_id = row[player_col]
        if pd.notna(player_id):
            y_value = row[y_col]
            if pd.isna(y_value):
                position = None
            elif y_value <= 1:
                position = "Goalkeeper"
            elif y_value <= 3:
                position = "Defender"
            elif y_value <= 8:
                position = "Midfielder"
            else:
                position = "Forwarder"
            info_on_player_list.append({"season": season,
                                      "team_api_id": row["away_team_api_id"],
                                      "player_api_id": player_id,
                                      "position": position})
df_player_info = pd.DataFrame(info_on_player_list)

## Determine most frequent position of a player per season (this is set to be the player's position)
df_position_counts = (df_player_info
                      .groupby(["season", "team_api_id", "player_api_id", "position"])
                      .size()
                      .reset_index(name="position_counts"))

df_main_position = (df_position_counts
                    .sort_values("position_counts", ascending=False)
                    .drop_duplicates(subset=["season", "player_api_id"]))

## Number of games played per player in a season  ##
# Needed for bubble sizes (alternative for missing information on minutes played)
df_total_games_played = (df_player_info
                         .groupby(["season", "player_api_id"])
                         .size()
                         .reset_index(name="games_played"))

# Combine total appearances with main position and team
df_gamesplayed = df_total_games_played.merge(df_main_position[["season", "player_api_id", "team_api_id", "position"]],
                                             on=["season", "player_api_id"],
                                             how="left")

## Goals ##
# Only counting valid goals (n and p)
df_goals = df_match_goals[df_match_goals["goal_type"].isin(["n", "p"])].copy()
df_goals = df_goals.merge(df_match[["id", "season"]], left_on="match_id", right_on="id", how="left")
df_goals_player = (df_goals
                   .groupby(["season", "player1"])
                   .size()
                   .reset_index(name="goals")
                   .rename(columns={"player1": "player_api_id"}))

## Assists ##
df_assists = (df_goals
              .dropna(subset=["player2"])
              .groupby(["season", "player2"])
              .size()
              .reset_index(name="assists")
              .rename(columns={"player2": "player_api_id"}))

## Combine different dataframes into one ##
df_complete = df_gamesplayed.merge(df_goals_player, on=["season", "player_api_id"], how="left")
df_complete = df_complete.merge(df_assists, on=["season", "player_api_id"], how="left")
df_complete = df_complete.merge(df_player[["player_api_id", "player_name"]], on="player_api_id", how="left")
df_complete = df_complete.merge(df_team[["team_api_id", "team_long_name"]], on="team_api_id", how="left")
# Fill missing data in goals and assists with zeros
df_complete["goals"] = df_complete["goals"].fillna(0)
df_complete["assists"] = df_complete["assists"].fillna(0)
# Drop data when there is no information on players position
df_complete = df_complete.dropna(subset=["position"])
# Exclude goalkeepers
df_complete = df_complete[df_complete["position"] != "Goalkeeper"].copy()
# Keep only players with goals or assists
df_complete = df_complete[(df_complete["goals"] > 0) | (df_complete["assists"] > 0)].copy()

## Team ranking for drop down by number of goals ##
df_team_rank = (df_complete
                .groupby(["season", "team_api_id"])["goals"]
                .sum()
                .reset_index())

df_team_rank["team_rank"] = (df_team_rank
                             .groupby("season")["goals"]
                             .rank(method="first", ascending=False))

## Generating Chart

In [7]:
''' Building Bubble Chart '''

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
seasons = sorted(df_complete["season"].dropna().unique())
app.layout = dbc.Container([
    html.H2("Match Involvement of Goal Scorers", className="my-4"),
    # Drop down for season
    html.Label("Select Season"),
        dcc.Dropdown(
        id="season-dropdown",
        options=[{"label": season, "value": season} for season in seasons],
        value=seasons[0],
        clearable=False,
        className="mb-3"),
    # Drop down for team selection
    html.Label("Select Team"),
    dcc.Dropdown(
        id="team-dropdown",
        value="all",
        clearable=False,
        className="mb-4"),
    dcc.Graph(id="bubble-chart")])

# Specifications for dropdown for team selection
@app.callback(Output("team-dropdown", "options"), Input("season-dropdown", "value"))
def update_team_dropdown(selected_season):
    df_filtered = df_complete[df_complete["season"] == selected_season]
    teams = sorted(df_filtered["team_long_name"]
                   .dropna()
                   .unique())
    menu_options = [{"label": "All Teams", "value": "all"},
               {"label": "Top 100 Teams", "value": "top100"},
               {"label": "Top 50 Teams", "value": "top50"},
               {"label": "Top 20 Teams", "value": "top20"},
               {"label": "Top 15 Teams", "value": "top15"},
               {"label": "Top 10 Teams", "value": "top10"}]
    menu_options += [{"label": team, "value": team} for team in teams]
    return menu_options

# Update plot after user makes a selection in drop-down menu 
@app.callback(Output("bubble-chart", "figure"), Input("season-dropdown", "value"), Input("team-dropdown", "value"))
def update_chart(selected_season, selected_team):
    df_plot = df_complete[df_complete["season"] == selected_season].copy()
    if selected_team != "all":
        if selected_team == "top100":
            top_teams = df_team_rank[(df_team_rank["season"] == selected_season) & 
                                     (df_team_rank["team_rank"] <= 100)]["team_api_id"]
            df_plot = df_plot[df_plot["team_api_id"].isin(top_teams)]
        elif selected_team == "top50":
            top_teams = df_team_rank[(df_team_rank["season"] == selected_season) &
                                     (df_team_rank["team_rank"] <= 50)]["team_api_id"]
            df_plot = df_plot[df_plot["team_api_id"].isin(top_teams)]
        elif selected_team == "top20":
            top_teams = df_team_rank[(df_team_rank["season"] == selected_season) &
                                     (df_team_rank["team_rank"] <= 20)]["team_api_id"]
            df_plot = df_plot[df_plot["team_api_id"].isin(top_teams)]
        elif selected_team == "top15":
            top_teams = df_team_rank[(df_team_rank["season"] == selected_season) &
                                     (df_team_rank["team_rank"] <= 15)]["team_api_id"]
            df_plot = df_plot[df_plot["team_api_id"].isin(top_teams)]
        elif selected_team == "top10":
            top_teams = df_team_rank[(df_team_rank["season"] == selected_season) &
                                     (df_team_rank["team_rank"] <= 10)]["team_api_id"]
            df_plot = df_plot[df_plot["team_api_id"].isin(top_teams)]
        else:
            df_plot = df_plot[df_plot["team_long_name"] == selected_team]

    fig = px.scatter(
        df_plot,
        x="goals",
        y="assists",
        size="games_played",
        color="position",
        # Set up for information that plops up for each bubble
        hover_name="player_name",
        custom_data=["position", "team_long_name", "games_played"],
        # Color scheme for different players position
        color_discrete_map={"Forwarder": "lightblue",
                            "Midfielder": "green",
                            "Defender": "orange"},
        title=f"Season: {selected_season}",
        labels={"goals": "Goals",
                "assists": "Goal Creating Action (Assists)",
                "position": "Player's Position"},
        size_max=30)
    # Specificiation of information that plops up for each bubble
    fig.update_traces(hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Position: %{customdata[0]}<br>"
        "Team: %{customdata[1]}<br>"
        "Goals: %{x}<br>"
        "Assists: %{y}<br>"
        "Games played: %{customdata[2]}"
        "<extra></extra>"))
    fig.update_layout(template="plotly_white")
    return fig
app.run(debug=True, port = 8090)